# TRACER — Module 7: Image Reconstruction

Runs real diffusion-based reconstruction on a real PGD-attacked image, guided by the real
attribution heatmap from Module 6, and reports real SSIM/PSNR fidelity against the true
clean original.

**This is the slowest notebook so far** — it downloads a multi-GB Stable Diffusion model.
Expect the model download alone to take several minutes, and each reconstruction call
~30-60 seconds on a T4. Get a coffee.

In [ ]:
GITHUB_USERNAME = "IqraYasmin123"   # change if this isn't your username
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/tracers.git"

import os
if not os.path.exists("/content/tracers"):
    !git clone {REPO_URL} /content/tracers
else:
    %cd /content/tracers
    !git pull

%cd /content/tracers/ai-engine
import sys
sys.path.insert(0, ".")

from src.vlm import CLIPVLM, VLMConfig
from src.dataset import FlickrDataset, DatasetConfig
from src.attacks import PGDAttack, AttackConfig
from src.attribution import GradientSaliency, AttentionMap
from src.reconstruction import (
    DiffusionReconstructor, ReconstructionConfig,
    compute_reconstruction_metrics, compare_images,
)
import torch
print("Imports OK.")


## Log into Hugging Face

Strongly recommended this time — the diffusion model download is much larger than CLIP's,
so rate-limit stalls (seen in Module 5) are more costly to hit here.

In [ ]:
from huggingface_hub import login
HF_TOKEN = ""  # paste your token here, e.g. "hf_..."
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("No token provided — continuing without login (may hit rate limits).")


## Load CLIP and Flickr8k (local-disk copy, verified via completion marker)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

vlm = CLIPVLM(VLMConfig())
print(f"CLIP loaded on {vlm.device}")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/TRACER/datasets/flickr8k"
LOCAL_DATASET_DIR = "/content/flickr8k_local"
COPY_MARKER = os.path.join(LOCAL_DATASET_DIR, ".copy_complete")

if not os.path.exists(COPY_MARKER):
    print("Copying Flickr8k to local disk...")
    !rm -rf "{LOCAL_DATASET_DIR}"
    !cp -r "{DRIVE_DATASET_DIR}" "{LOCAL_DATASET_DIR}"
    with open(COPY_MARKER, "w") as f:
        f.write("done")
    print("Copy complete.")
else:
    print("Local copy already present and verified complete.")

dataset = FlickrDataset(DatasetConfig(root_dir=LOCAL_DATASET_DIR, image_size=(224, 224)))
print(f"Loaded {len(dataset)} (image, caption) pairs.")


## Pick a sample, attack it with PGD, and generate an attribution heatmap

In [ ]:
sample = dataset[0]
real_image = sample["image"]
true_caption = sample["caption"]

pixel_values = vlm.preprocess_image(real_image)
text_embeds = vlm.encode_text([true_caption])

pgd_adv = PGDAttack(AttackConfig(epsilon=8/255, alpha=2/255, steps=10)).generate(
    vlm, pixel_values, text_embeds
)

clean_sim = vlm.similarity(vlm.encode_image(pixel_values), text_embeds).item()
adv_sim = vlm.similarity(vlm.encode_image(pgd_adv), text_embeds).item()
print(f"Caption: {true_caption}")
print(f"Clean similarity: {clean_sim:.4f}  |  PGD adversarial similarity: {adv_sim:.4f}")

attribution = GradientSaliency().generate(vlm, pgd_adv, text_embeds)
print(f"Attribution heatmap generated (method: {attribution.method})")


## Convert the adversarial tensor back to a real image

The diffusion pipeline needs a PIL image, not a normalized tensor — undo CLIP's
normalization to get back a real, displayable image.

In [ ]:
def to_pil(pixel_values, processor, device):
    mean = torch.tensor(processor.image_processor.image_mean).view(1, 3, 1, 1).to(device)
    std = torch.tensor(processor.image_processor.image_std).view(1, 3, 1, 1).to(device)
    img = (pixel_values * std + mean).clamp(0, 1)
    arr = (img.squeeze().permute(1, 2, 0).detach().cpu().numpy() * 255).astype("uint8")
    from PIL import Image
    return Image.fromarray(arr)

adversarial_pil = to_pil(pgd_adv, vlm.processor, vlm.device)
adversarial_pil


## Load the diffusion pipeline and reconstruct

**This cell downloads a multi-GB model — expect several minutes the first time.** Uses the
real heatmap from above (resized to match the image) as the inpainting mask.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

pipeline = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16
).to(vlm.device)

reconstructor = DiffusionReconstructor(
    config=ReconstructionConfig(mask_threshold=0.6),
    pipeline=pipeline,
)

heatmap_for_mask = attribution.resized((adversarial_pil.height, adversarial_pil.width))
reconstructed = reconstructor.reconstruct(adversarial_pil, heatmap_for_mask)
print("Reconstruction complete.")


## Visual comparison: clean vs. adversarial vs. reconstructed

In [ ]:
fig = compare_images(real_image, adversarial_pil, reconstructed)


## Reconstruction fidelity metrics (vs. the true clean original)

In [ ]:
metrics = compute_reconstruction_metrics(real_image, reconstructed)
print(f"SSIM: {metrics['SSIM']:.4f}  (1.0 = identical to clean original)")
print(f"PSNR: {metrics['PSNR']:.2f} dB  (higher is better; >30 dB is generally considered good)")

# For comparison: how far the ADVERSARIAL image already was from clean, before reconstruction
adv_metrics = compute_reconstruction_metrics(real_image, adversarial_pil)
print(f"\n(For reference) Clean vs. adversarial (no reconstruction) SSIM: {adv_metrics['SSIM']:.4f}, PSNR: {adv_metrics['PSNR']:.2f} dB")


## Module 7 completion check

Run the full test suite:
```bash
cd ai-engine
pytest tests/test_reconstruction.py -v
```
All 14 tests should pass. Combined with the real SSIM/PSNR numbers above and the visual
3-way comparison, Module 7 is complete — and with it, TRACER's full detect → localize →
reconstruct pipeline has been demonstrated end-to-end on real data. Continue to
**Module 8 — Explainable AI (XAI)**.